## Install packages

In [1]:
!pip install -q pandas pyarrow sqlalchemy psycopg2-binary minio python-dotenv

## Import

In [2]:
import os
import ast
from pathlib import Path
from io import BytesIO

import pandas as pd
from dotenv import load_dotenv
from minio import Minio
from sqlalchemy import create_engine

## Config MinIO + PostgreSQL

In [3]:
# ===============================
# BASE PATH
# ===============================
BASE_DIR = Path("/home/jovyan/work")
ENV_FILE = BASE_DIR / ".env"

load_dotenv(ENV_FILE)


# ===============================
# MINIO CONFIG
# ===============================
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "admin")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "password123")
MINIO_BUCKET = os.getenv("MINIO_BUCKET", "linkedin-data")

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False,
)


# ===============================
# POSTGRES CONFIG
# ===============================
POSTGRES_USER = os.getenv("POSTGRES_USER", "admin")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "password123")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "postgres")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POSTGRES_DB = os.getenv("POSTGRES_DB", "job_mart")

ENGINE_URL = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)

engine = create_engine(ENGINE_URL)

print("MinIO endpoint:", MINIO_ENDPOINT)
print("MinIO bucket  :", MINIO_BUCKET)
print("PostgreSQL    :", f"{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")
print("Config ready")

MinIO endpoint: minio:9000
MinIO bucket  : bigdata-nhom6
PostgreSQL    : postgres:5432/job_mart
Config ready


## Test connection

In [4]:
# Test MinIO
if not client.bucket_exists(MINIO_BUCKET):
    raise ValueError(f"Bucket không tồn tại: {MINIO_BUCKET}")

print("MinIO connection OK")


# Test PostgreSQL
with engine.connect() as conn:
    result = conn.exec_driver_sql("SELECT current_database();").fetchone()[0]
    print("PostgreSQL connection OK")
    print("Current database:", result)

MinIO connection OK
PostgreSQL connection OK
Current database: job_mart


## Define MinIO paths

In [5]:
# ===============================
# GOLD INPUT
# ===============================
GOLD_OBJECT = "gold/kaggle/model_input.parquet"


# ===============================
# MART OUTPUT
# ===============================
MART_PREFIX = "mart/kaggle/dashboard"

MART_OBJECTS = {
    "mart_jobs": f"{MART_PREFIX}/mart_jobs.parquet",
    "mart_job_skills": f"{MART_PREFIX}/mart_job_skills.parquet",
    "mart_title_skill_summary": f"{MART_PREFIX}/mart_title_skill_summary.parquet",
    "mart_location_skill_summary": f"{MART_PREFIX}/mart_location_skill_summary.parquet",
}

print("Gold object:", GOLD_OBJECT)
print("Mart prefix:", MART_PREFIX)

Gold object: gold/kaggle/model_input.parquet
Mart prefix: mart/kaggle/dashboard


## Helper functions for MinIO parquet

In [6]:
def read_parquet_from_minio(object_name: str) -> pd.DataFrame:
    response = client.get_object(MINIO_BUCKET, object_name)

    try:
        data = response.read()
    finally:
        response.close()
        response.release_conn()

    return pd.read_parquet(BytesIO(data))


def write_parquet_to_minio(df: pd.DataFrame, object_name: str) -> None:
    buffer = BytesIO()
    df.to_parquet(buffer, index=False)
    buffer.seek(0)

    client.put_object(
        bucket_name=MINIO_BUCKET,
        object_name=object_name,
        data=buffer,
        length=buffer.getbuffer().nbytes,
        content_type="application/octet-stream",
    )

    print(f"Uploaded: s3://{MINIO_BUCKET}/{object_name}")


def list_minio_objects(prefix: str):
    objects = client.list_objects(
        MINIO_BUCKET,
        prefix=prefix,
        recursive=True
    )

    return [obj.object_name for obj in objects]

## Check Gold file exists on MinIO

In [7]:
gold_objects = list_minio_objects("gold/kaggle/")

print("Objects under gold/kaggle/:")
for obj in gold_objects[:20]:
    print("-", obj)

if GOLD_OBJECT not in gold_objects:
    print("\nWARNING: GOLD_OBJECT chưa thấy trong danh sách trên.")
    print("Hãy kiểm tra lại tên file Gold trên MinIO.")
else:
    print("\nGold file found.")

Objects under gold/kaggle/:
- gold/kaggle/model_input.parquet

Gold file found.


## Load Gold Kaggle data

In [8]:
gold_df = read_parquet_from_minio(GOLD_OBJECT)

print("Gold shape:", gold_df.shape)
print("Gold columns:")
print(gold_df.columns.tolist())

gold_df.head()

Gold shape: (762185, 5)
Gold columns:
['job_link', 'job_title', 'location', 'skills', 'embedding_text']


,job_link,job_title,location,skills,embedding_text
0,https://www.linkedin.com/jobs/view/account-exe...,account executive,united states,"[nursing, pharmacy]",account executive in united states
1,https://www.linkedin.com/jobs/view/registered-...,registered nurse,united states,"[nursing, patient education]",registered nurse in united states
2,https://www.linkedin.com/jobs/view/restaurant-...,restaurant supervisor,united states,[inventory management],restaurant supervisor in united states
3,https://www.linkedin.com/jobs/view/registered-...,registered nurse,united states,"[bsn, medical license, nursing]",registered nurse in united states
4,https://uk.linkedin.com/jobs/view/store-manage...,store manager,united kingdom,[kpi management],store manager in united kingdom


## Parse skills + validate source data

In [30]:
import re
import ast
import sys

sys.path.append(str(BASE_DIR))

from recommendation.core.preprocess import (
    process_job_title,
    normalize_location,
    normalize_skill,
)


# =========================================================
# ROLE KEYWORDS
# Chỉ giữ job_title có ít nhất 1 dấu hiệu là nghề thật
# =========================================================
ROLE_KEYWORDS = {
    # General corporate / tech roles
    "analyst", "developer", "engineer", "manager", "consultant",
    "specialist", "coordinator", "administrator", "architect",
    "designer", "scientist", "programmer", "technician",
    "supervisor", "director", "officer", "associate",
    "assistant", "lead", "head", "principal", "senior", "junior",
    "intern", "trainee",

    # Healthcare
    "nurse", "physician", "doctor", "therapist", "pharmacist",
    "dentist", "clinician", "caregiver", "anesthesiologist",
    "hygienist", "surgeon", "practitioner", "radiologist",
    "counselor", "psychologist",

    # Finance / accounting
    "accountant", "auditor", "controller", "finance", "financial",
    "bookkeeper", "payroll", "tax", "treasurer",

    # Education
    "teacher", "instructor", "professor", "tutor", "educator",

    # Business / sales / customer
    "sales", "marketing", "recruiter", "recruitment",
    "customer", "support", "service", "representative",
    "executive", "advisor", "agent",

    # Operations / blue-collar / service
    "driver", "mechanic", "operator", "electrician", "plumber",
    "chef", "cook", "server", "cashier", "clerk", "warehouse",
    "assembler", "installer", "inspector",

    # Data / software / IT
    "product", "project", "program", "operations", "business",
    "data", "software", "cloud", "security", "network",
    "database", "system", "systems", "devops", "frontend",
    "backend", "full", "stack", "qa", "quality", "assurance",
}


ROLE_PHRASES = [
    "data analyst",
    "business analyst",
    "financial analyst",
    "software engineer",
    "backend developer",
    "back end developer",
    "front end developer",
    "frontend developer",
    "full stack developer",
    "project manager",
    "product manager",
    "program manager",
    "account executive",
    "registered nurse",
    "sales representative",
    "customer service",
    "machine learning",
    "quality assurance",
    "general dentist",
    "dental hygienist",
    "nurse practitioner",
    "care manager",
    "care coordinator",
    "data scientist",
    "data engineer",
    "business intelligence",
]


# =========================================================
# BLACKLISTS
# =========================================================
BAD_EXACT_TITLES = {
    "", "na", "n a", "n/a", "none", "null", "other", "unknown",
    "remote", "hybrid", "onsite", "on site",
    "full time", "part time", "contract", "temporary", "temp",
    "shift", "day shift", "night shift", "evening shift",
    "morning shift", "rotating shift", "weekend shift",
    "variable", "schedule", "availability",
    "staff", "worker", "employee", "team member",
}

BAD_FRAGMENTS = {
    "0 0", "0 01", "shift", "variable", "schedule",
    "weekend", "hourly", "per diem", "opening",
    "opportunity", "hiring", "wanted", "urgent", "immediate",
}

LOCATION_LIKE = {
    # US states
    "ca", "ny", "tx", "fl", "ga", "ma", "wa", "nj", "il", "pa",
    "az", "co", "oh", "mi", "nc", "va", "tn", "mo", "mn",
    "al", "ak", "ar", "ct", "de", "hi", "ia", "id", "in",
    "ks", "ky", "la", "md", "me", "ms", "mt", "nd", "ne",
    "nh", "nm", "nv", "ok", "or", "ri", "sc", "sd", "ut",
    "vt", "wi", "wv", "wy",

    # Common city/location fragments
    "covington", "dallas", "houston", "austin", "seattle",
    "chicago", "boston", "atlanta", "miami", "phoenix",
    "denver", "new york", "los angeles", "san francisco",
    "san diego", "san jose", "orlando", "tampa", "remote",
}

# Các từ ngành/khoa/phòng ban, nếu đứng một mình hoặc không có role thì loại
BAD_DOMAIN_ONLY_TERMS = {
    "anesthesiology", "cardiology", "radiology", "dermatology",
    "neurology", "oncology", "pediatrics", "orthopedics",
    "pathology", "psychiatry", "surgery", "medicine",
    "dental", "medical", "healthcare", "moffitt",
}


# =========================================================
# SKILL PARSER
# =========================================================
def parse_skills(value):
    # Case 1: list / tuple / set
    if isinstance(value, (list, tuple, set)):
        return [
            normalize_skill(str(x).strip())
            for x in value
            if normalize_skill(str(x).strip())
        ]

    # Case 2: numpy array / pandas array
    if hasattr(value, "tolist"):
        try:
            value_list = value.tolist()

            if isinstance(value_list, list):
                return [
                    normalize_skill(str(x).strip())
                    for x in value_list
                    if normalize_skill(str(x).strip())
                ]
        except Exception:
            pass

    # Case 3: missing scalar
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    # Case 4: string
    if isinstance(value, str):
        value = value.strip()

        if not value:
            return []

        # String dạng "['sql', 'python']"
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, list):
                return [
                    normalize_skill(str(x).strip())
                    for x in parsed
                    if normalize_skill(str(x).strip())
                ]
        except Exception:
            pass

        # String dạng "sql, python, power bi"
        return [
            normalize_skill(x.strip())
            for x in value.split(",")
            if normalize_skill(x.strip())
        ]

    return []


# =========================================================
# TITLE CLEANING
# =========================================================
def remove_leading_noise_tokens(title):
    """
    Loại prefix/code ngắn ở đầu title:
    - a3 general dentist -> general dentist
    - aa registered nurse care manager -> registered nurse care manager
    - 01 data analyst -> data analyst
    """
    if not isinstance(title, str):
        return ""

    tokens = title.lower().strip().split()

    while tokens:
        first = tokens[0]

        # token toàn số
        if first.isdigit():
            tokens.pop(0)
            continue

        # token quá ngắn như aa, ca, ny
        if len(first) <= 2:
            tokens.pop(0)
            continue

        # token dạng a3, b2, x1
        if re.match(r"^[a-z]{1,2}[0-9]{1,2}$", first):
            tokens.pop(0)
            continue

        break

    return " ".join(tokens).strip()


def hard_clean_title(title):
    if not isinstance(title, str):
        return ""

    title = title.lower().strip()

    # đổi ký tự phân tách thành khoảng trắng
    title = re.sub(r"[_\\|/]+", " ", title)

    # bỏ phần trong ngoặc
    title = re.sub(r"\(.*?\)", " ", title)
    title = re.sub(r"\[.*?\]", " ", title)

    # bỏ ký tự đặc biệt ở đầu/cuối
    title = re.sub(r"^[^a-zA-Z0-9]+", "", title)
    title = re.sub(r"[^a-zA-Z0-9]+$", "", title)

    # clean bằng hàm cũ
    title = process_job_title(title)

    # clean lại
    title = re.sub(r"[^a-zA-Z0-9\s\-/]", " ", title)
    title = re.sub(r"\s+", " ", title).strip()

    # bỏ prefix/code ngắn ở đầu
    title = remove_leading_noise_tokens(title)

    return title


def has_role_keyword(title):
    if not isinstance(title, str):
        return False

    title = title.lower().strip()
    words = set(title.split())

    # keyword đơn
    if words & ROLE_KEYWORDS:
        return True

    # phrase nghề nghiệp
    if any(phrase in title for phrase in ROLE_PHRASES):
        return True

    return False


def is_valid_job_title(title):
    if not isinstance(title, str):
        return False

    title = title.lower().strip()

    if not title:
        return False

    words = title.split()
    word_count = len(words)

    # quá ngắn / quá dài
    if word_count < 2:
        return False

    if word_count > 8:
        return False

    # phải có chữ cái
    if not re.search(r"[a-z]", title):
        return False

    # không bắt đầu bằng số hoặc ký tự đặc biệt
    if re.match(r"^[0-9]", title):
        return False

    if re.match(r"^[^a-zA-Z]", title):
        return False

    # token đầu vẫn là code ngắn thì loại
    first_token = words[0]

    if len(first_token) <= 2:
        return False

    if re.match(r"^[a-z]{1,2}[0-9]{1,2}$", first_token):
        return False

    # quá nhiều số bất thường
    digit_count = sum(ch.isdigit() for ch in title)
    if digit_count >= 2:
        return False

    # blacklist exact
    if title in BAD_EXACT_TITLES:
        return False

    # title giống location
    if title in LOCATION_LIKE:
        return False

    # chứa fragment rác
    if any(fragment in title for fragment in BAD_FRAGMENTS):
        return False

    # chỉ là domain/khoa/phòng ban nhưng không có role thật
    title_words = set(title.split())

    if (title_words & BAD_DOMAIN_ONLY_TERMS) and not has_role_keyword(title):
        return False

    # phải có dấu hiệu là nghề thật
    if not has_role_keyword(title):
        return False

    return True


# =========================================================
# BUILD CLEAN SOURCE
# =========================================================
df = gold_df.copy()

required_cols = ["job_title", "location", "skills"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

if "job_link" not in df.columns:
    df["job_link"] = ""

before_all = len(df)

# Parse + normalize skills
df["skills"] = df["skills"].apply(parse_skills)

# Clean job title
df["job_title_raw"] = df["job_title"]
df["job_title"] = df["job_title"].apply(hard_clean_title)

# Clean location
df["location"] = df["location"].apply(normalize_location)

# Clean job_link
df["job_link"] = df["job_link"].fillna("").astype(str).str.strip()

# Basic filter
before_basic = len(df)

df = df[
    df["job_title"].notna()
    & df["location"].notna()
    & df["skills"].apply(lambda x: len(x) > 0)
].copy()

df["job_title"] = df["job_title"].astype(str).str.strip()
df["location"] = df["location"].astype(str).str.strip()

df = df[
    (df["job_title"] != "")
    & (df["location"] != "")
].copy()

after_basic = len(df)

# Strong title validation
before_title_filter = len(df)

df["is_valid_title"] = df["job_title"].apply(is_valid_job_title)

invalid_titles_df = df[~df["is_valid_title"]][
    ["job_title_raw", "job_title", "location", "skills"]
].copy()

df = df[df["is_valid_title"]].drop(columns=["is_valid_title"]).copy()

after_title_filter = len(df)

print("Rows before all filters      :", before_all)
print("Rows after basic filter      :", after_basic)
print("Removed by basic filter      :", before_basic - after_basic)
print("Removed invalid job titles   :", before_title_filter - after_title_filter)
print("Final clean source shape     :", df.shape)

print("\nSample invalid titles removed:")
display(invalid_titles_df.head(30))

print("\nSample clean source:")
display(df[["job_title_raw", "job_title", "location", "skills"]].head(30))

Rows before all filters      : 762185
Rows after basic filter      : 747095
Removed by basic filter      : 15090
Removed invalid job titles   : 196319
Final clean source shape     : (550776, 6)

Sample invalid titles removed:


,job_title_raw,job_title,location,skills
9,human resource generalist,human resource generalist,united states,"[adp, microsoft office, sap]"
12,travel technologist,travel technologist,united states,"[computed tomography, equipment maintenance, h..."
21,diversité équité et inclusion,diversit quit et inclusion,canada,"[excel, microsoft office suite, powerpoint, tr..."
27,ultrasound technologist,ultrasound technologist,united states,"[cpr, hippa compliance, mammography, patient c..."
29,department manager 37 000 yr,department manager 37,united states,"[crew scheduling, inventory management]"
30,technician,technician,united states,[autocad]
31,designer,designer,united states,"[indesign, photoshop]"
34,beauty advisor sally beauty 03500,beauty advisor sally beauty 03500,united states,[inventory management]
38,electrical engineer 23,electrical engineer 23,united states,"[cabling, circuit design, electrical engineeri..."
39,travel technologist,travel technologist,united states,[computed tomography]



Sample clean source:


,job_title_raw,job_title,location,skills
0,account executive,account executive,united states,"[nursing, pharmacy]"
1,registered nurse,registered nurse,united states,"[nursing, patient education]"
2,restaurant supervisor,restaurant supervisor,united states,[inventory management]
3,registered nurse,registered nurse,united states,"[bsn, medical license, nursing]"
4,store manager,store manager,united kingdom,[kpi management]
5,engineering project coordinator,engineering project coordinator,canada,"[autocad, construction management, engineering]"
6,assistant vice president,assistant vice president,united states,[nursing]
7,senior sale associate,senior sale associate,united states,"[flatbed scanner operation, planogram implemen..."
8,senior associate tax,senior associate tax,united states,"[microsoft office suite, sql, uml, xml]"
10,executive director hospice,executive director hospice,united states,"[cpr certification, hospice care, nursing, pat..."


## Create 4 Mart tables

In [31]:
# ===============================
# 1. mart_jobs
# ===============================
mart_source = df[["job_link", "job_title", "location", "skills"]].copy()
mart_source = mart_source.reset_index(drop=True)
mart_source["job_id"] = mart_source.index + 1

mart_jobs = mart_source.copy()
mart_jobs["skill_count"] = mart_jobs["skills"].apply(len)
mart_jobs["skills_text"] = mart_jobs["skills"].apply(lambda x: ", ".join(x))

mart_jobs = mart_jobs[
    [
        "job_id",
        "job_link",
        "job_title",
        "location",
        "skill_count",
        "skills_text",
    ]
].copy()


# ===============================
# 2. mart_job_skills
# ===============================
mart_job_skills = (
    mart_source[["job_id", "job_title", "location", "skills"]]
    .explode("skills")
    .rename(columns={"skills": "skill"})
)

mart_job_skills = mart_job_skills[
    mart_job_skills["skill"].notna()
    & (mart_job_skills["skill"].astype(str).str.strip() != "")
].copy()

mart_job_skills["skill"] = mart_job_skills["skill"].astype(str).str.strip()

mart_job_skills = mart_job_skills[
    [
        "job_id",
        "job_title",
        "location",
        "skill",
    ]
].copy()


# ===============================
# 3. mart_title_skill_summary
# ===============================
mart_title_skill_summary = (
    mart_job_skills
    .groupby(["job_title", "skill"], as_index=False)
    .agg(job_count=("job_id", "nunique"))
    .sort_values(["job_title", "job_count"], ascending=[True, False])
)


# ===============================
# 4. mart_location_skill_summary
# ===============================
mart_location_skill_summary = (
    mart_job_skills
    .groupby(["location", "skill"], as_index=False)
    .agg(job_count=("job_id", "nunique"))
    .sort_values(["location", "job_count"], ascending=[True, False])
)


print("mart_jobs:", mart_jobs.shape)
print("mart_job_skills:", mart_job_skills.shape)
print("mart_title_skill_summary:", mart_title_skill_summary.shape)
print("mart_location_skill_summary:", mart_location_skill_summary.shape)

mart_jobs: (550776, 6)
mart_job_skills: (1861947, 4)
mart_title_skill_summary: (691800, 3)
mart_location_skill_summary: (6556, 3)


## Preview Mart tables

In [32]:
display(mart_jobs.head())
display(mart_job_skills.head())
display(mart_title_skill_summary.head())
display(mart_location_skill_summary.head())

,job_id,job_link,job_title,location,skill_count,skills_text
0,1,https://www.linkedin.com/jobs/view/account-exe...,account executive,united states,2,"nursing, pharmacy"
1,2,https://www.linkedin.com/jobs/view/registered-...,registered nurse,united states,2,"nursing, patient education"
2,3,https://www.linkedin.com/jobs/view/restaurant-...,restaurant supervisor,united states,1,inventory management
3,4,https://www.linkedin.com/jobs/view/registered-...,registered nurse,united states,3,"bsn, medical license, nursing"
4,5,https://uk.linkedin.com/jobs/view/store-manage...,store manager,united kingdom,1,kpi management


,job_id,job_title,location,skill
0,1,account executive,united states,nursing
0,1,account executive,united states,pharmacy
1,2,registered nurse,united states,nursing
1,2,registered nurse,united states,patient education
2,3,restaurant supervisor,united states,inventory management


,job_title,skill,job_count
0,aac specialist,assistive technology,1
1,aac specialist,occupational therapy,1
2,aalto supervisor,quality control,1
3,aat qualified part qualified accountant,aat,1
4,aat qualified part qualified accountant,acca,1


,location,skill,job_count
834,australia,microsoft office suite,970
42,australia,ahpra registration,818
502,australia,excel,722
154,australia,budgeting,709
902,australia,nursing,626


## Basic data quality checks

In [33]:
print("Total jobs:", mart_jobs["job_id"].nunique())
print("Total job-skill rows:", len(mart_job_skills))
print("Unique titles:", mart_jobs["job_title"].nunique())
print("Unique locations:", mart_jobs["location"].nunique())
print("Unique skills:", mart_job_skills["skill"].nunique())


print("\nTop 10 titles:")
top_titles = (
    mart_jobs["job_title"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_titles.columns = ["job_title", "job_count"]
display(top_titles)


print("\nTop 10 skills:")
top_skills_df = (
    mart_job_skills["skill"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_skills_df.columns = ["skill", "job_count"]
display(top_skills_df)


print("\nTop 10 locations:")
top_locations = (
    mart_jobs["location"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_locations.columns = ["location", "job_count"]
display(top_locations)

Total jobs: 550776
Total job-skill rows: 1861947
Unique titles: 108648
Unique locations: 5
Unique skills: 1888

Top 10 titles:


,job_title,job_count
0,registered nurse,25458
1,senior sale associate,7558
2,assistant manager,6909
3,store manager,5986
4,travel registered nurse,3754
5,senior accountant,3427
6,senior software engineer,2744
7,project manager,2729
8,maintenance supervisor,2318
9,senior project manager,2305



Top 10 skills:


,skill,job_count
0,nursing,52948
1,patient care,49368
2,microsoft office suite,46784
3,excel,45878
4,inventory management,41775
5,microsoft office,36568
6,budgeting,31953
7,troubleshooting,25987
8,accounting,24170
9,quality assurance,21351



Top 10 locations:


,location,job_count
0,united states,471255
1,united kingdom,46037
2,canada,21412
3,australia,12071
4,vietnam,1


## Save Mart tables to MinIO

In [34]:
write_parquet_to_minio(
    mart_jobs,
    MART_OBJECTS["mart_jobs"]
)

write_parquet_to_minio(
    mart_job_skills,
    MART_OBJECTS["mart_job_skills"]
)

write_parquet_to_minio(
    mart_title_skill_summary,
    MART_OBJECTS["mart_title_skill_summary"]
)

write_parquet_to_minio(
    mart_location_skill_summary,
    MART_OBJECTS["mart_location_skill_summary"]
)


print("\nMart objects on MinIO:")
for obj in list_minio_objects(MART_PREFIX):
    print("-", obj)

Uploaded: s3://bigdata-nhom6/mart/kaggle/dashboard/mart_jobs.parquet
Uploaded: s3://bigdata-nhom6/mart/kaggle/dashboard/mart_job_skills.parquet
Uploaded: s3://bigdata-nhom6/mart/kaggle/dashboard/mart_title_skill_summary.parquet
Uploaded: s3://bigdata-nhom6/mart/kaggle/dashboard/mart_location_skill_summary.parquet

Mart objects on MinIO:
- mart/kaggle/dashboard/mart_job_skills.parquet
- mart/kaggle/dashboard/mart_jobs.parquet
- mart/kaggle/dashboard/mart_location_skill_summary.parquet
- mart/kaggle/dashboard/mart_title_skill_summary.parquet


## Save Mart tables to PostgreSQ

In [35]:
mart_jobs.to_sql(
    "mart_jobs",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10000,
)

mart_job_skills.to_sql(
    "mart_job_skills",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10000,
)

mart_title_skill_summary.to_sql(
    "mart_title_skill_summary",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10000,
)

mart_location_skill_summary.to_sql(
    "mart_location_skill_summary",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10000,
)

print("Saved 4 mart tables to PostgreSQL")

Saved 4 mart tables to PostgreSQL


## Create indexes in PostgreSQL

In [36]:
index_sql = [
    "CREATE INDEX IF NOT EXISTS idx_mart_jobs_job_id ON mart_jobs(job_id);",
    "CREATE INDEX IF NOT EXISTS idx_mart_jobs_title ON mart_jobs(job_title);",
    "CREATE INDEX IF NOT EXISTS idx_mart_jobs_location ON mart_jobs(location);",

    "CREATE INDEX IF NOT EXISTS idx_mart_job_skills_job_id ON mart_job_skills(job_id);",
    "CREATE INDEX IF NOT EXISTS idx_mart_job_skills_skill ON mart_job_skills(skill);",
    "CREATE INDEX IF NOT EXISTS idx_mart_job_skills_title ON mart_job_skills(job_title);",
    "CREATE INDEX IF NOT EXISTS idx_mart_job_skills_location ON mart_job_skills(location);",

    "CREATE INDEX IF NOT EXISTS idx_title_skill_title ON mart_title_skill_summary(job_title);",
    "CREATE INDEX IF NOT EXISTS idx_title_skill_skill ON mart_title_skill_summary(skill);",

    "CREATE INDEX IF NOT EXISTS idx_location_skill_location ON mart_location_skill_summary(location);",
    "CREATE INDEX IF NOT EXISTS idx_location_skill_skill ON mart_location_skill_summary(skill);",
]

with engine.begin() as conn:
    for sql in index_sql:
        conn.exec_driver_sql(sql)

print("Indexes created")

Indexes created


## Verify PostgreSQL tables

In [37]:
tables = [
    "mart_jobs",
    "mart_job_skills",
    "mart_title_skill_summary",
    "mart_location_skill_summary",
]

with engine.connect() as conn:
    for table in tables:
        count = conn.exec_driver_sql(
            f"SELECT COUNT(*) FROM {table};"
        ).fetchone()[0]

        print(f"{table}: {count:,} rows")

mart_jobs: 550,776 rows
mart_job_skills: 1,861,947 rows
mart_title_skill_summary: 691,800 rows
mart_location_skill_summary: 6,556 rows
